In [3]:
#=====================================================================
  run_ubicacion
  Barrido de las 576 combinaciones (nodo BESS x nodo Eolico) para
  encontrar la ubicacion optima
=====================================================================#
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX, Printf

# ---------- RUTAS DE ARCHIVOS ----------
const SCRIPT_DIR = @__DIR__                    # .../UC MODEL/SCRIPTS
const BASE    = dirname(SCRIPT_DIR)            # .../UC MODEL
const IN_BESS = joinpath(BASE, "INPUT", "BESS")
const IN_WIND = joinpath(BASE, "INPUT", "Wind power generation forecast")
const IN_NODE = joinpath(BASE, "INPUT", "24 NODE MODEL")
const OUT_LOC = joinpath(BASE, "OUTPUT", "LOCATION")
mkpath(OUT_LOC)
println("BASE del proyecto: ", BASE)

# ---------- DATOS ----------
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
buses_gen      = Int64.(sheet["B"][2:end]); Pmax = Float64.(sheet["C"][2:end]); Pmin = Float64.(sheet["D"][2:end])
Costo          = Float64.(sheet["E"][2:end]); Costofijo = Float64.(sheet["F"][2:end]); CostoArranque = Float64.(sheet["G"][2:end])
nGen = length(Pmax)

sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
demanda_horaria = Float64.(sheet_demanda["B"][2:end]); T = length(demanda_horaria)

sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2]); A_vec = Float64.(sheet_eolico["D"][2]); Cp_vec = Float64.(sheet_eolico["E"][2])

sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
velocidades_viento = Float64.(sheet_viento["B"][2:end])
function potencia_eolica_teorica(v)
    if v < 3 || v > 25
        return 0.0
    elseif v <= 11.8
        return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
    else
        return 1.5
    end
end
potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]

sheet_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))[1]
nLineas = count(!ismissing, sheet_lineas["A"][2:end])
desde = Int64.(sheet_lineas["A"][2:1+nLineas]); hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
reactancia = Float64.(sheet_lineas["C"][2:1+nLineas]); flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
nBuses = maximum(vcat(desde, hasta))

sheet_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))[1]
Emax=Float64(sheet_bess["B"][2]); pev_max=Float64(sheet_bess["C"][2])
η_c=Float64(sheet_bess["D"][2]); η_d=Float64(sheet_bess["E"][2]); ramp_pev=Float64(sheet_bess["F"][2])
SOC_ini=Float64(sheet_bess["G"][2]); SOC_min_pct=Float64(sheet_bess["H"][2]); pen_soc_low=Float64(sheet_bess["I"][2])
carga_bess=Float64(sheet_bess["J"][2]); descarga_bess=Float64(sheet_bess["K"][2]); degradacion_bess=Float64(sheet_bess["L"][2])

sheet_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))[1]
buses_carga = Int64.(sheet_carga["A"][2:end]); porcentajes_carga = Float64.(sheet_carga["B"][2:end])
distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))

# ---------- MODELO ----------
function evaluar_modelo(bus_bess::Int, bus_eolico::Int)
    model = Model(HiGHS.Optimizer); set_silent(model)
    @variable(model, p[g=1:nGen, t=1:T]); @variable(model, u[g=1:nGen, t=1:T], Bin)
    @variable(model, v[g=1:nGen, t=1:T], Bin); @variable(model, θ[b=1:nBuses, t=1:T])
    @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t])
    @variable(model, 0 <= charge[t=1:T] <= pev_max); @variable(model, 0 <= discharge[t=1:T] <= pev_max)
    @variable(model, 0 <= soc[t=1:T] <= Emax); @variable(model, pen_low[t=1:T] >= 0); @variable(model, z[t=1:T], Bin)
    for t in 1:T, b in 1:nBuses
        gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
        gen += (b == bus_eolico ? pw[t] : 0.0); gen += (b == bus_bess ? discharge[t] : 0.0)
        demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
        if b == bus_bess; demanda_b += charge[t]; end
        inflow = sum((θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
        outflow = sum((θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        @constraint(model, gen + inflow - outflow == demanda_b)
    end
    for g in 1:nGen, t in 1:T
        @constraint(model, p[g,t] <= Pmax[g] * u[g,t]); @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
    end
    for g in 1:nGen
        @constraint(model, v[g,1] >= u[g,1])
        for t in 2:T; @constraint(model, v[g,t] >= u[g,t] - u[g,t-1]); end
    end
    @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
    for t in 2:T
        @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
        @constraint(model, charge[t] - charge[t-1] <= ramp_pev); @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
        @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev); @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
    end
    for t in 1:T; @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t]); end
    for t in 1:T
        @constraint(model, charge[t] <= pev_max * (1 - z[t])); @constraint(model, discharge[t] <= pev_max * z[t])
    end
    for (i,j) in keys(x), t in 1:T
        fij = (θ[i,t] - θ[j,t]) / x[(i,j)]
        @constraint(model, fij <= Fmax[(i,j)]); @constraint(model, fij >= -Fmax[(i,j)])
    end
    for t in 1:T; @constraint(model, θ[1,t] == 0); end
    @constraint(model, soc[T] >= SOC_ini / 100 * Emax)
    @objective(model, Min,
        sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T) +
        sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T) + sum(pen_low[t]*pen_soc_low for t=1:T) +
        sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T) +
        sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T))
    optimize!(model)
    st = termination_status(model)
    return (st == MOI.OPTIMAL || st == MOI.FEASIBLE_POINT) ? objective_value(model) : Inf
end

# ---------- BARRIDO ----------
open(joinpath(OUT_LOC,"OUT_sweep.csv"),"w") do io
    println(io,"bus_bess,bus_eolico,costo")
    t0 = time(); cont = 0
    for bess_nodo in 1:nBuses, eolico_nodo in 1:nBuses
        c = evaluar_modelo(bess_nodo, eolico_nodo)
        @printf(io,"%d,%d,%.4f\n", bess_nodo, eolico_nodo, c); cont += 1
        if cont % 24 == 0; @printf("Progreso: %d/%d  (%.1f s)\n", cont, nBuses*nBuses, time()-t0); flush(stdout); end
    end
    @printf("BARRIDO COMPLETO en %.1f s\n", time()-t0)
end

# ---------- TOP 10 ----------
using DelimitedFiles
rows = readdlm(joinpath(OUT_LOC,"OUT_sweep.csv"), ',', skipstart=1)
costos = rows[:,3]; validos = costos .< Inf
optc = minimum(costos[validos]); orden = sortperm(costos)
open(joinpath(OUT_LOC,"top10.csv"),"w") do io
    println(io,"#,bus_bess,bus_eolico,costo,sobrecosto_pct")
    for k in 1:10
        idx = orden[k]; c = costos[idx]
        @printf(io,"%d,%d,%d,%.4f,%.4f\n", k, Int(rows[idx,1]), Int(rows[idx,2]), c, 100*(c-optc)/optc)
    end
end
imin = orden[1]
@printf("\n>> Optimo: BESS nodo %d | Eolico nodo %d | costo %.2f\n", Int(rows[imin,1]), Int(rows[imin,2]), optc)
println("Resultados en: ", OUT_LOC)


BASE del proyecto: C:\TESIS_UC\ENTREGABLES\UC MODEL
Progreso: 24/576  (37.5 s)
Progreso: 48/576  (66.9 s)
Progreso: 72/576  (117.6 s)
Progreso: 96/576  (146.8 s)
Progreso: 120/576  (174.6 s)
Progreso: 144/576  (204.1 s)
Progreso: 168/576  (229.4 s)
Progreso: 192/576  (256.7 s)
Progreso: 216/576  (281.8 s)
Progreso: 240/576  (322.0 s)
Progreso: 264/576  (351.4 s)
Progreso: 288/576  (380.8 s)
Progreso: 312/576  (402.7 s)
Progreso: 336/576  (434.0 s)
Progreso: 360/576  (467.8 s)
Progreso: 384/576  (492.7 s)
Progreso: 408/576  (514.1 s)
Progreso: 432/576  (539.6 s)
Progreso: 456/576  (566.6 s)
Progreso: 480/576  (594.2 s)
Progreso: 504/576  (623.9 s)
Progreso: 528/576  (650.1 s)
Progreso: 552/576  (674.9 s)
Progreso: 576/576  (702.0 s)
BARRIDO COMPLETO en 702.0 s

>> Optimo: BESS nodo 1 | Eolico nodo 1 | costo 429870.38
Resultados en: C:\TESIS_UC\ENTREGABLES\UC MODEL\OUTPUT\LOCATION
